# MapBiomas Fire Catalog - U-Net test - Brazil - Pantanal

**Campaign:** `brazil.collection_06` | **Sensor:** landsat | **Periodicity:** yearly | **Region:** pantanal

Alternative to M4/M5 (pixel MLP): a **U-Net** trained on image chips. It reuses
the **same mosaics** (M2 COGs) and the **same `fire`/`notFire` polygons** (M3 CSVs)
as `collection_06_brazil.ipynb`. The MLP flow is not changed.

Code: `network/catalog/development/version_04/src/unet/` (branch `classification-unet`).
Guide: `src/unet/README.md`.

**Use a fresh runtime with GPU** (Runtime > Change runtime type > GPU). Do not run
the M4 trainer in the same runtime: it switches TensorFlow to v1 mode.

#### Steps

| Step | What it does | Output (GCS `.../COLLECTION_06/UNET_01/`) |
|---|---|---|
| 0 | Setup, authentication, U-Net config | - |
| 1 | Inventory: mosaic grid and samples | printed report |
| 2 | Spatial blocks + chips + labels (1 fire / 0 not fire / 255 unknown) | `chips/`, `splits/`, `reports/chips_*` |
| 3 | Normalization statistics (train chips only) | `stats/norm_stats.json` |
| 4 | Train the U-Net | `models/` |
| 5 | Test metrics (threshold chosen on validation) | `reports/metrics_*.json` |
| 6 | Classify the whole region | `predict/pantanal_{year}_prob.tif` |
| 7 | Compare with the MLP (A bands, B bands + indices, C U-Net) | `reports/comparison_*.json` |

## [0] Setup

In [ ]:
# 0.1 -- Colab setup -- Brazil | landsat | yearly | COLLECTION_06 | U-Net
import sys, os, subprocess, shutil

BRANCH = "classification-unet"
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)

subprocess.run(["apt-get", "update", "-qq"])
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"])
subprocess.run(["pip", "install", "-q", "earthengine-api", "gcsfs", "rasterio", "geopandas", "shapely>=2", "pyproj"])

V04 = os.path.join(REPO_DIR, "network/catalog/development/version_04")
for p in (os.path.join(V04, "src/unet"), os.path.join(V04, "src/core")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(os.path.join(V04, "src/core"))

from M0_auth_config import set_global_opts, authenticate, print_config

COUNTRY_SELECTED = "brazil"
CAMPAIGN_SELECTED = "COLLECTION_06"
set_global_opts(
    country=COUNTRY_SELECTED,
    campaign=CAMPAIGN_SELECTED,
    sensor="landsat",
    periodicity="yearly",
    mosaic_methods="minnbr",
    personal_task_flag="FUEGO",
    language="pt",
    gcs_bucket="mapbiomas-fire",
    gcs_library_images_prefix=f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    gcs_campaigns_prefix=f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    gee_project=f"mapbiomas-{COUNTRY_SELECTED}",
    gee_library_images_prefix=f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    gee_campaigns_prefix=f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    asset_regions=f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01/{CAMPAIGN_SELECTED}/AUXILIARY/regiones_fuego_{COUNTRY_SELECTED}_v1",
)
authenticate()
print_config()

In [ ]:
# 0.2 -- U-Net configuration (every path comes from here + M0)
from unet_config import make_config, paths

UCFG = make_config(
    MODEL="unet",                    # "mlp" -> use collection_06_brazil.ipynb (M4/M5)
    EXPERIMENT="UNET_01",            # GCS folder inside the campaign
    MODEL_NAME="unet_pantanal_v1",
    REGION="pantanal",               # `region_nam` in the regions asset and `region` in the samples
    YEARS=[2024],
    BANDS=["blue", "red", "nir", "swir1", "swir2"],
    INDICES=["nbr", "nbr2", "mirbi", "ndvi", "bai", "evi"],
    SAMPLE_GLOB="*{year}*.csv",      # M3 CSVs in LIBRARY_SAMPLES; or SAMPLE_FILES=[...]
    CHIP=128, POINTS_PER_POLYGON=4, BLOCK_KM=25, SPLIT=(0.70, 0.15, 0.15),
    BATCH=16, EPOCHS=60, LR=1e-4, SEED=42,
    OVERLAP=16, INFER_TILE=512,
)
P = paths(UCFG)
if UCFG["MODEL"] != "unet":
    raise SystemExit("MODEL='mlp': run M4/M5 in collection_06_brazil.ipynb.")
print(f"Outputs: gs://{P['bucket']}/{P['root']}/")
print(f"Samples: gs://{P['bucket']}/{P['samples']}/")

## [1] Inventory (read only)
Checks that every band is on the same grid and summarizes the samples.

In [ ]:
import json
from unet_io import MosaicReader, load_samples, list_sample_files

for year in UCFG["YEARS"]:
    reader = MosaicReader(UCFG, year)
    print(json.dumps(reader.inventory(), indent=2, default=str))
    print("Sample files:", list_sample_files(UCFG, year))
    samples, report = load_samples(UCFG, year, reader.crs)
    print(json.dumps(report["by_class"], indent=2))
    reader.close()

## [2] Chips and labels
Blocks of `BLOCK_KM` are split into train/val/test **before** the chips are made.
A chip that crosses blocks of different splits is dropped. Label: 1 fire, 0 not fire,
255 outside the polygons or nodata (ignored by the loss). Fire wins where polygons overlap.

In [ ]:
from IPython.display import Image, display
from unet_chips import build_chips
from unet_io import local_dir

for year in UCFG["YEARS"]:
    summary = build_chips(UCFG, year, overwrite=False)
    if not summary.get("skipped"):
        print(json.dumps({k: summary[k] for k in ("blocks", "chips", "dropped", "fire_notfire_overlaps")},
                         indent=2, default=str)[:3000])
        png = os.path.join(local_dir("reports"), f"chips_preview_{year}.png")
        if os.path.exists(png):
            display(Image(png))

Check the preview: red = fire, blue = not fire. Look for chips over lagoons and wet areas (dark water looks like a burn scar).

## [3] Normalization statistics (train chips only)

In [ ]:
from unet_chips import compute_stats
stats = compute_stats(UCFG)
for name, m, s in zip(stats["channels"], stats["mean"], stats["std"]):
    print(f"{name:6s} mean {m:9.4f} std {s:9.4f}")

## [4] Train the U-Net

In [ ]:
from unet_model import train
model, meta = train(UCFG)
print("epochs:", meta["epochs_run"], "| best val IoU:", max(meta["history"]["val_iou_fire"]))

## [5] Test metrics
Threshold with the best F1 on **validation**; the test is reported at that threshold and at 0.5.

In [ ]:
from unet_model import load_model
from unet_report import evaluate_unet
model = load_model(UCFG)
unet_results = evaluate_unet(UCFG, model)
print(json.dumps(unet_results["all"]["test_at_best"], indent=2))

## [6] Classify the whole region
Sliding window: each window reads `INFER_TILE + 2*OVERLAP` pixels and keeps only the
center, so there are no seams. Output in the M5 format (probability x 1000 + dayOfYear,
nodata -9999), same grid as the mosaic.

In [ ]:
from unet_io import read_json
from unet_predict import predict_mosaic, gee_upload_command
stats = read_json(P["stats"])
for year in UCFG["YEARS"]:
    res = predict_mosaic(UCFG, year, model, stats)
    print(res)
    # Optional: upload to GEE (edit the asset path first)
    print(gee_upload_command(UCFG, year, f"projects/mapbiomas-brazil/assets/FIRE/CATALOG_01/COLLECTION_06/UNET_01/pantanal_{year}_prob"))

## [7] Comparison with the MLP
Same test pixels for every model. A = MLP bands, B = MLP bands + indices, C = U-Net.

In [ ]:
from IPython.display import Markdown
from unet_report import compare_with_mlp
rows, table = compare_with_mlp(UCFG, unet_results)
display(Markdown(table))